# 00 - GenoTools QC and TOPMed imputation (non-NBA cohorts only)

Run this **only** for cohorts that are not part of the GP2 release: their raw
arrays need sample and variant QC, ancestry assignment and imputation before
they can enter the GWAS. Release cohorts (NBA and WGS) arrive QCed and imputed,
so they skip straight to `01_prep_inputs_nba.ipynb` /
`02_prep_inputs_wgs.ipynb`.

Steps: fill in sex and phenotype, keep the samples that have both, assign
ancestry against 1000 Genomes, run the GenoTools standard variant and sample QC pipeline, then
prepare and submit the result to TOPMed.

Most cells print a command rather than running it - the heavy steps belong in
a terminal (or under `nohup`), not in a kernel that may be restarted.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

import gwas_config as cfg
from genotools.imputation import impute_data_prep

pd.options.mode.chained_assignment = None

cfg.describe()

## One-off setup

```
git clone https://github.com/dvitale199/GenoTools.git
git clone https://github.com/hirotaka-i/1kg_ref.git
```

Reference files expected in `cfg.REF_DIR`: `hg38.fa.gz` (+ `.fai`),
`hg19ToHg38.over.chain.gz`, `hg38ToHg19.over.chain.gz`, the filtered 1000
Genomes panel with its population labels, and - for the imputation prep - the
TOPMed panel and `HRC-1000G-check-bim.pl`.

`liftOver` must be on the PATH:

```
wget http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64.v479/liftOver
chmod +x liftOver && sudo mv liftOver /usr/local/bin/
```

## Parameters

In [ ]:
COHORT = 'OPDC'
ANCESTRY = 'EUR'

COHORT_DIR = cfg.WORK_MOUNT / cfg.PREFIX / 'non_NBA' / COHORT
RAW_GENO = COHORT_DIR / f'{COHORT}_raw'    # genotypes as received (bed/bim/fam)
SCRATCH = cfg.TMP_DIR / COHORT             # fast local scratch, cleared with the VM

COHORT_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
print(COHORT_DIR)

## Fill in sex and phenotype

The arrays usually arrive without either, and the QC steps need both.

In [ ]:
sex_file = COHORT_DIR / 'clinical_info' / f'{COHORT}_update_sex.txt'
pheno_file = COHORT_DIR / 'clinical_info' / f'{COHORT}_update_phenotype.txt'
geno_path = COHORT_DIR / COHORT

print(cfg.shell_cmd(f'plink2 --bfile {RAW_GENO}',
                    f'--update-sex {sex_file}',
                    f'--pheno {pheno_file}',
                    f'--make-pgen --out {geno_path}'))

In [ ]:
psam = pd.read_csv(f'{geno_path}.psam', sep=r'\s+')
display(psam.SEX.value_counts(dropna=False))
display(psam.PHENO1.value_counts(dropna=False))
print(f'{len(psam[psam.SEX.isnull() | psam.PHENO1.isnull()])} samples missing one of them')

pvar = pd.read_csv(f'{geno_path}.pvar', sep=r'\s+', low_memory=False)
print(f'{len(pvar)} variants')
display(pvar['#CHROM'].value_counts())

In [ ]:
# Keep only samples that have both a sex and a phenotype.
keep = psam[psam.SEX.notnull() & psam.PHENO1.notnull()]
keep_file = COHORT_DIR / f'keep_{COHORT}.samples'
keep[['#FID', 'IID']].to_csv(keep_file, sep='\t', index=False, header=None)
print(f'keeping {len(keep)} of {len(psam)} samples')

subset_path = COHORT_DIR / f'{COHORT}_subset'
print(cfg.shell_cmd(f'plink2 --pfile {geno_path}',
                    f'--keep {keep_file}',
                    f'--make-pgen --out {subset_path}'))

## Ancestry assignment

Lift to hg38, QC the samples, then project onto 1000 Genomes and split the
cohort by assigned population. Run these from a directory where `./bin/plink2`
resolves (e.g. `/usr/local`).

In [ ]:
fasta = cfg.REF_DIR / 'hg38.fa.gz'
qc = cfg.QC_CODE_DIR / 'code'

print(cfg.shell_cmd(f'bash {qc}/01_prepare_variants.sh',
                    f'--pfile {subset_path}',
                    f'--fa {fasta}',
                    f'--chain {cfg.REF_DIR}/hg19ToHg38.over.chain.gz',
                    f'--chain-back {cfg.REF_DIR}/hg38ToHg19.over.chain.gz',
                    f'--outdir {SCRATCH}/prep_vars_out',
                    '--threads 2'))

In [ ]:
print(cfg.shell_cmd(f'bash {qc}/02_qc_samples.sh',
                    f'--pfile {SCRATCH}/prep_vars_out/hg38_prepped',
                    f'--outdir {SCRATCH}/qc_samples_out',
                    '--threads 2'))

In [ ]:
print(cfg.shell_cmd(f'bash {qc}/03_pca_with_ref_score.sh',
                    f'--study-pfile {SCRATCH}/qc_samples_out/final_keep',
                    f'--ref-bfile {cfg.REF_DIR}/all_hg38_filtered_chrpos',
                    f'--fa {fasta}',
                    '--drop-ambig yes',
                    f'--outdir {SCRATCH}/merge_ref_proj_out',
                    '--threads 2'))

In [ ]:
print(cfg.shell_cmd(f'python {qc}/03b_plot_pop_and_split.py',
                    f'--pc-prefix {SCRATCH}/merge_ref_proj_out/study_vs_ref.combined',
                    f'--ref-label {cfg.REF_DIR}/all_hg38_filtered_chrpos_pop.txt',
                    '--ref-label-col Population',
                    '--split-method mahalanobis',
                    f'--out-prefix {SCRATCH}/pop_split_out/with_1kg_mah'))

In [ ]:
# Samples per assigned ancestry. Only run QC for groups big enough to
# estimate their own PCs - roughly n > 50.
ancestry_dir = COHORT_DIR / f'{COHORT}_ancestry'
for path in sorted(ancestry_dir.glob('*.list')):
    n = len(pd.read_csv(path, header=None, sep='\t'))
    print(f'{path.name:45} {n:6}')

## GenoTools standard pipeline
Subset to ancestries of interest based on outputs from above steps

In [ ]:
inputs = cfg.TMP_DIR / 'inputs'
inputs.mkdir(parents=True, exist_ok=True)

anc_keep = ancestry_dir / f'with_1kg_mah_{ANCESTRY}.list'
anc_pfile = inputs / f'{COHORT}_subset_{ANCESTRY}'

print(cfg.shell_cmd(f'plink2 --pfile {subset_path}',
                    f'--keep {anc_keep}',
                    f'--make-pgen --out {anc_pfile}'))

In [ ]:
# Lift the ancestry subset to hg38 before QC.
print(cfg.shell_cmd(f'bash {qc}/01_prepare_variants.sh',
                    f'--pfile {anc_pfile}',
                    f'--fa {fasta}',
                    f'--chain {cfg.REF_DIR}/hg19ToHg38.over.chain.gz',
                    f'--chain-back {cfg.REF_DIR}/hg38ToHg19.over.chain.gz',
                    f'--outdir {inputs}',
                    '--threads 2'))

In [ ]:
qc_input = inputs / 'hg38_prepped'
qc_output = COHORT_DIR / f'{COHORT}_output' / f'{COHORT}_cleaned_{ANCESTRY}'
qc_output.parent.mkdir(parents=True, exist_ok=True)

genotools_cmd = cfg.shell_cmd(
    f'python {cfg.GENOTOOLS_DIR}/genotools',
    f'--pfile {qc_input}',
    f'--out {qc_output}',
    '--ancestry False',      # ancestry was assigned above
    '--full_output False',
    '--all_sample True',
    '--all_variant True')
print(f'nohup {genotools_cmd} > genotools_{COHORT}_{ANCESTRY}.log 2>&1 &')

## Review the QC output

In [ ]:
pvar = pd.read_csv(f'{qc_output}.pvar', sep=r'\s+', low_memory=False)
print(f'{len(pvar)} variants after QC')
display(pvar['#CHROM'].value_counts())

with open(f'{qc_output}.json') as f:
    data = json.load(f)

display(pd.DataFrame(data['QC']))              # what each step removed
display(pd.DataFrame(data['pruned_samples']))  # which samples removed

## Prepare for TOPMed and submit

`impute_data_prep` runs the HRC/1000G check script against the TOPMed panel and
writes one pre-imputation VCF per chromosome; those are indexed and checked
against the reference before submission.

In [ ]:
impute_prep_dir = COHORT_DIR / 'imputation_prep' / ANCESTRY
imputed_out_dir = COHORT_DIR / 'imputation_out_dir' / ANCESTRY
impute_prep_dir.mkdir(parents=True, exist_ok=True)
imputed_out_dir.mkdir(parents=True, exist_ok=True)

# The prep script works on PLINK 1.9 bed/bim/fam.
print(f'plink2 --pfile {qc_output} --make-bed --out {qc_output}')

In [ ]:
topmed_ref_panel = cfg.REF_DIR / 'PASS.Variants.TOPMed_freeze5_hg38_dbSNP.tab.gz'
check_bim_pl = cfg.REF_DIR / 'HRC-1000G-check-bim.pl'

impute_data_prep(str(qc_output), str(impute_prep_dir / ANCESTRY),
                 str(topmed_ref_panel), str(check_bim_pl))

In [ ]:
for chrom in range(1, 23):
    vcf = impute_prep_dir / f'{ANCESTRY}_pre_impute_chr{chrom}.vcf.gz'
    fixed = vcf.with_name(vcf.name.replace('.vcf.gz', '_fixed.vcf.gz'))
    cfg.shell_do(f'tabix -p vcf {vcf} -f')
    cfg.shell_do(f'bcftools norm --check-ref s --fasta-ref {cfg.REF_DIR}/hg38.fa '
                 f'{vcf} -Oz -o {fixed}')

In [ ]:
# Every chromosome should be present
for chrom in range(1, 23):
    fixed = impute_prep_dir / f'{ANCESTRY}_pre_impute_chr{chrom}_fixed.vcf.gz'
    size = f'{fixed.stat().st_size / 1e6:8.1f} MB' if fixed.exists() else '   MISSING'
    print(f'{fixed.name:48} {size}')

In [ ]:
# Check imputationbot is installed and authenticated, then submit
cfg.shell_do('imputationbot refpanels', log=True)

print(cfg.shell_cmd(
    'imputationbot impute',
    f'--files {impute_prep_dir}/{ANCESTRY}_pre_impute_chr*fixed.vcf.gz',
    '--refpanel topmed-r3 --build hg38 --r2Filter 0.3 --population all',
    f'--name {COHORT}_{ANCESTRY}'))

## Download the imputed genotypes

The job id and password come from the TOPMed job page

In [ ]:
job_id = os.environ.get('TOPMED_JOB_ID', '<job id>')

print(f'cd {imputed_out_dir} && '
      f'imputationbot download {job_id} --password "$TOPMED_JOB_PASSWORD"')

Normalise the downloaded dosages into per-chromosome pgen files named
`<cohort>_chr<N>_norm.pgen` under `<working mount>/<prefix>/non_NBA/<cohort>/PLINK/`
- that is where `03_prep_inputs_non_nba.ipynb` and `04_launch_gwas.ipynb` look
for them. Then continue with `03_prep_inputs_non_nba.ipynb`.